Daniel Kollar-Gasiewski

# FashionMNIST Image Classifier

Run the cells below in order.

## 0. Setup

Install the third-party dependencies into the current kernel. `torch` and
`torchvision` are usually already available, but `torchmetrics` often is not, so
install it here to make the notebook self-contained. If you have already
installed these, this cell is a quick no-op.

In [1]:
# Install dependencies into the kernel that is running this notebook.
# %pip installs into the *current* kernel, which avoids the common
# "installed in the wrong environment" problem with plain !pip.
%pip install -q torch torchvision torchmetrics

Note: you may need to restart the kernel to use updated packages.


## 1. Load the data

Load FashionMNIST with TorchVision into a training/validation dataset and a
test dataset, then split the train/validation data randomly (seed 42) into
55,000 training and 5,000 validation samples.

In [2]:
import torch
from torch.utils.data import random_split, DataLoader
import torchvision
from torchvision import transforms

# Convert the PIL images to tensors.
transform = transforms.ToTensor()

# Training/validation dataset (60,000 images).
train_val_dataset = torchvision.datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

# Test dataset (10,000 images).
test_dataset = torchvision.datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Split the 60,000 train/val images into 55,000 train and 5,000 validation,
# using a fixed seed of 42 for reproducibility.
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    train_val_dataset,
    [55_000, 5_000],
    generator=generator,
)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

Training samples:   55000
Validation samples: 5000
Test samples:       10000


## 2. Create the DataLoaders

Wrap the datasets in DataLoaders with a batch size of 32. The training loader
is shuffled (seed 42 for a reproducible batch order); the validation and test
loaders are not shuffled.

In [3]:
BATCH_SIZE = 32

# Seed the generator that drives the training loader's shuffling so the
# batch order is reproducible.
loader_generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

# Sanity check: inspect the shape of one training batch.
images, labels = next(iter(train_loader))
print(f"Image batch shape:  {tuple(images.shape)}")
print(f"Label batch shape:  {tuple(labels.shape)}")

Batch size: 32
Training batches:   1719
Validation batches: 157
Test batches:       313
Image batch shape:  (32, 1, 28, 28)
Label batch shape:  (32,)


## 3. Sample the data

Grab the first (x, y) sample from the training data to inspect the shape and
data type of an image, and use the class labels to see the FashionMNIST
classes shared by the training and validation data.

In [4]:
# Take the first sample from the training data.
x_sample, y_sample = train_dataset[0]

# Shape and data type of the x (image) sample.
print(f"x sample shape: {tuple(x_sample.shape)}")
print(f"x sample dtype: {x_sample.dtype}")

# The y sample is a class index; the class names live on the underlying
# FashionMNIST dataset. train_dataset and val_dataset are both Subsets of the
# same train/validation dataset, so they share the same classes.
train_classes = train_dataset.dataset.classes
val_classes = val_dataset.dataset.classes

print(f"\ny sample label index: {y_sample} -> {train_classes[y_sample]}")
print(f"\nTraining classes:   {train_classes}")
print(f"Validation classes: {val_classes}")

x sample shape: (1, 28, 28)
x sample dtype: torch.float32

y sample label index: 9 -> Ankle boot

Training classes:   ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Validation classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## 4. Build the image classifier

Define an `ImageClassifier` as a fully-connected (MLP) network. The constructor
takes the flattened `input_size` (28 x 28 = 784), a list of `hidden_sizes`, and
the number of output `num_classes` (10), and builds an `nn.Sequential` model
from those inputs. Then initialize a model and set up the loss function.

In [5]:
import torch.nn as nn


class ImageClassifier(nn.Module):
    """A simple fully-connected (MLP) image classifier.

    Args:
        input_size:   Number of input features per image after flattening
                      (e.g. 28 * 28 = 784 for FashionMNIST).
        hidden_sizes: List of hidden-layer widths. One Linear + ReLU block is
                      created for each entry.
        num_classes:  Number of output classes (10 for FashionMNIST).
    """

    def __init__(self, input_size, hidden_sizes, num_classes):
        super().__init__()

        # Build the layers for the Sequential model. Flatten first so the model
        # accepts image tensors of shape (batch, 1, 28, 28) directly.
        layers = [nn.Flatten()]
        in_features = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size
        # Final layer maps to the class logits (no activation; CrossEntropyLoss
        # applies softmax internally).
        layers.append(nn.Linear(in_features, num_classes))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


# Inputs for FashionMNIST: 1 x 28 x 28 images flattened to 784, and 10 classes.
input_size = 1 * 28 * 28
hidden_sizes = [300, 100]
num_classes = 10

# Initialize the model.
model = ImageClassifier(input_size, hidden_sizes, num_classes)

# Loss function.
loss_fn = nn.CrossEntropyLoss()

print(model)
print(f"\nLoss function: {loss_fn}")

ImageClassifier(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=300, bias=True)
    (2): ReLU()
    (3): Linear(in_features=300, out_features=100, bias=True)
    (4): ReLU()
    (5): Linear(in_features=100, out_features=10, bias=True)
  )
)

Loss function: CrossEntropyLoss()


## 5. Optimizer and training

Use an SGD optimizer to update the model's parameters. Then define a training
function that ties together the model, data loaders, loss function, and
optimizer, and train for 20 epochs. Each epoch reports the training and
validation loss and accuracy.

In [6]:
from torchmetrics.classification import MulticlassAccuracy

# SGD optimizer (learning rate 0.01).
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# torchmetrics accuracy metrics (one per split) for 10 FashionMNIST classes.
# average="micro" gives overall correct / total (standard accuracy).
train_accuracy = MulticlassAccuracy(num_classes=num_classes, average="micro")
val_accuracy = MulticlassAccuracy(num_classes=num_classes, average="micro")


def train(model, train_loader, val_loader, loss_fn, optimizer,
          train_accuracy, val_accuracy, num_epochs=20):
    """Train the model, reporting train/validation metrics each epoch.

    Ties together everything built in the previous cells: the model, the
    training and validation DataLoaders, the loss function, the optimizer, and
    the torchmetrics accuracy metrics.

    Returns a history dict of per-epoch loss and accuracy for both splits.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, num_epochs + 1):
        # --- Training pass ---
        model.train()
        train_accuracy.reset()
        train_loss, train_total = 0.0, 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_total += labels.size(0)
            train_accuracy.update(outputs, labels)

        train_loss /= train_total
        train_acc = train_accuracy.compute().item()

        # --- Validation pass ---
        model.eval()
        val_accuracy.reset()
        val_loss, val_total = 0.0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images)
                loss = loss_fn(outputs, labels)

                val_loss += loss.item() * images.size(0)
                val_total += labels.size(0)
                val_accuracy.update(outputs, labels)

        val_loss /= val_total
        val_acc = val_accuracy.compute().item()

        # Record metrics for this epoch.
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:2d}/{num_epochs}  "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    return history


# Train for 20 epochs and keep the metric history.
history = train(
    model, train_loader, val_loader, loss_fn, optimizer,
    train_accuracy, val_accuracy, num_epochs=20,
)

Epoch  1/20  train_loss=1.1160 train_acc=0.6267  val_loss=0.6895 val_acc=0.7614
Epoch  2/20  train_loss=0.5952 train_acc=0.7918  val_loss=0.5488 val_acc=0.8090
Epoch  3/20  train_loss=0.5061 train_acc=0.8236  val_loss=0.4993 val_acc=0.8242
Epoch  4/20  train_loss=0.4653 train_acc=0.8375  val_loss=0.4830 val_acc=0.8322
Epoch  5/20  train_loss=0.4396 train_acc=0.8448  val_loss=0.4600 val_acc=0.8390
Epoch  6/20  train_loss=0.4221 train_acc=0.8515  val_loss=0.4629 val_acc=0.8352
Epoch  7/20  train_loss=0.4071 train_acc=0.8568  val_loss=0.4244 val_acc=0.8496
Epoch  8/20  train_loss=0.3938 train_acc=0.8610  val_loss=0.4991 val_acc=0.8114
Epoch  9/20  train_loss=0.3818 train_acc=0.8650  val_loss=0.4103 val_acc=0.8564
Epoch 10/20  train_loss=0.3706 train_acc=0.8699  val_loss=0.3993 val_acc=0.8572
Epoch 11/20  train_loss=0.3619 train_acc=0.8723  val_loss=0.4155 val_acc=0.8576
Epoch 12/20  train_loss=0.3525 train_acc=0.8755  val_loss=0.3769 val_acc=0.8670
Epoch 13/20  train_loss=0.3443 train_acc

## 6. Evaluate the model

Run the trained model over the validation loader to get the prediction logits,
take the class with the maximum logit as the prediction, and compare against
the true labels to see which predictions were correct.

In [7]:
# Put the model in evaluation mode and run over the validation data.
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in val_loader:
        # Prediction logits for the batch: shape (batch_size, num_classes).
        logits = model(images)
        # Take the class with the maximum logit as the prediction.
        preds = logits.argmax(dim=1)

        all_preds.append(preds)
        all_labels.append(labels)

# Combine the per-batch results into single tensors.
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

# Check which predictions were correct.
correct = all_preds == all_labels
num_correct = correct.sum().item()
total = correct.numel()

print(f"Correct predictions: {num_correct} / {total}")
print(f"Validation accuracy: {num_correct / total:.4f}")

# Peek at the first 10 predictions vs. the true labels.
print(f"\nFirst 10 predictions: {all_preds[:10].tolist()}")
print(f"First 10 labels:      {all_labels[:10].tolist()}")
print(f"First 10 correct?:    {correct[:10].tolist()}")

Correct predictions: 4358 / 5000
Validation accuracy: 0.8716

First 10 predictions: [7, 4, 2, 5, 9, 8, 7, 7, 7, 4]
First 10 labels:      [7, 4, 2, 5, 9, 8, 7, 9, 7, 4]
First 10 correct?:    [True, True, True, True, True, True, True, False, True, True]


## 7. Softmax probabilities and model size

Take the softmax of a sample's logits with `torch.nn.functional`, pull out the
top 4 probabilities and their class indices, and count the total number of
parameters in the model using `numel()`.

In [8]:
import torch.nn.functional as F

# Take a single sample's logits from the validation data.
images, labels = next(iter(val_loader))
model.eval()
with torch.no_grad():
    logits = model(images)

# Logits for the first sample in the batch.
y_logits = logits[0]

# Softmax turns the logits into class probabilities.
probs = F.softmax(y_logits, dim=0)

# Top 4 probabilities and their class indices.
top4_values, top4_indices = torch.topk(probs, 4)

print(f"Softmax probabilities: {probs.tolist()}")
print(f"\nTop 4 values:  {top4_values.tolist()}")
print(f"Top 4 indices: {top4_indices.tolist()}")
print(f"Top 4 classes: {[train_classes[i] for i in top4_indices.tolist()]}")

# Sum up all the parameters in the model using numel().
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal model parameters: {total_params:,}")

Softmax probabilities: [9.052010864252225e-05, 7.560220183222555e-06, 2.586898517620284e-05, 0.00012709830480162054, 9.824875633057673e-06, 0.027467604726552963, 3.550261681084521e-05, 0.9302472472190857, 0.005279507953673601, 0.03670918941497803]

Top 4 values:  [0.9302472472190857, 0.03670918941497803, 0.027467604726552963, 0.005279507953673601]
Top 4 indices: [7, 9, 5, 8]
Top 4 classes: ['Sneaker', 'Ankle boot', 'Sandal', 'Bag']

Total model parameters: 266,610
